# Notebook 1 — Data Cleaning
### CDC Foodborne Disease Outbreak Surveillance System (1998–2015)

**Goal:** Transform the raw CDC dataset into a structurally clean, analysis-ready file.  
**Input:** `outbreaks.csv` — 19,119 records × 12 columns  
**Output:** `cleaned_data.csv` — 18,828 records × 9 columns

**Steps:**
1. Initial audit — shape, dtypes, missingness, duplicates
2. Drop irrelevant columns
3. Remove duplicates
4. Handle missing values (categorical → `'Unknown'`, numerical → `0.0`)
5. Final validation and export

## 1. Setup

In [1]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("data/outbreaks.csv")
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)

Shape: 19,119 rows × 12 columns


,Year,Month,State,Location,Food,Ingredient,Species,Serotype/Genotype,Status,Illnesses,Hospitalizations,Fatalities
0,1998,January,California,Restaurant,NaN,NaN,NaN,NaN,NaN,20,0.0,0.0
1,1998,January,California,NaN,Custard,NaN,NaN,NaN,NaN,112,0.0,0.0
2,1998,January,California,Restaurant,NaN,NaN,NaN,NaN,NaN,35,0.0,0.0


## 2. Initial audit

Understanding what we have before making any changes.

In [2]:
# Data types and non-null counts
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19119 entries, 0 to 19118
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Year               19119 non-null  int64  
 1   Month              19119 non-null  object 
 2   State              19119 non-null  object 
 3   Location           16953 non-null  object 
 4   Food               10156 non-null  object 
 5   Ingredient         1876 non-null   object 
 6   Species            12500 non-null  object 
 7   Serotype/Genotype  3907 non-null   object 
 8   Status             12500 non-null  object 
 9   Illnesses          19119 non-null  int64  
 10  Hospitalizations   15494 non-null  float64
 11  Fatalities         15518 non-null  float64
dtypes: float64(2), int64(2), object(8)
memory usage: 1.8+ MB


In [3]:
# Missingness — absolute counts and percentage
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
audit = pd.DataFrame({"missing_n": missing, "missing_%": missing_pct,
                       "unique_values": df_raw.nunique()})
audit[audit["missing_n"] > 0].sort_values("missing_%", ascending=False)

,missing_n,missing_%,unique_values
Ingredient,17243,90.2,381
Serotype/Genotype,15212,79.6,239
Food,8963,46.9,3127
Species,6619,34.6,201
Status,6619,34.6,22
Hospitalizations,3625,19.0,61
Fatalities,3601,18.8,12
Location,2166,11.3,161


In [4]:
# Duplicate check
n_dupes = df_raw.duplicated().sum()
print(f"Duplicate rows: {n_dupes} ({n_dupes/len(df_raw)*100:.1f}% of dataset)")

Duplicate rows: 280 (1.5% of dataset)


**Audit summary:**

| Column | Missing | Decision |
|---|---|---|
| `Ingredient` | ~90% | Drop — too sparse to use |
| `Serotype/Genotype` | ~80% | Drop — outside EDA scope |
| `Status` | ~35% | Drop — administrative field |
| `Food` | 47% | Fill → `'Unknown'` |
| `Species` | 34% | Fill → `'Unknown'` |
| `Location` | 11% | Fill → `'Unknown'` |
| `Hospitalizations` | 19% | Fill → `0.0` (conservative) |
| `Fatalities` | 19% | Fill → `0.0` (conservative) |

**Missingness strategy rationale:** Replacing with `'Unknown'` rather than dropping rows
preserves ~47% of Food records and ~34% of Species records for temporal and geographic
analysis — where knowing *when* and *where* an outbreak occurred matters even if the
pathogen/food is unconfirmed.

## 3. Drop irrelevant columns

In [5]:
df = df_raw.copy()
df = df.drop(columns=["Ingredient", "Serotype/Genotype", "Status"])
print(f"Columns after drop: {df.columns.tolist()}")
print(f"Shape: {df.shape}")

Columns after drop: ['Year', 'Month', 'State', 'Location', 'Food', 'Species', 'Illnesses', 'Hospitalizations', 'Fatalities']
Shape: (19119, 9)


## 4. Remove duplicates

In [6]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed: {before - after} duplicate rows  ({before:,} → {after:,})")

Removed: 291 duplicate rows  (19,119 → 18,828)


## 5. Handle missing values

In [7]:
# Categorical columns → 'Unknown'
for col in ["Location", "Food", "Species"]:
    n_filled = df[col].isna().sum()
    df[col] = df[col].fillna("Unknown")
    print(f"{col}: filled {n_filled:,} missing values → 'Unknown'")

Location: filled 2,151 missing values → 'Unknown'
Food: filled 8,697 missing values → 'Unknown'
Species: filled 6,390 missing values → 'Unknown'


In [8]:
# Numerical columns → 0.0
for col in ["Hospitalizations", "Fatalities"]:
    n_filled = df[col].isna().sum()
    df[col] = df[col].fillna(0.0)
    print(f"{col}: filled {n_filled:,} missing values → 0.0")

Hospitalizations: filled 3,560 missing values → 0.0
Fatalities: filled 3,538 missing values → 0.0


## 6. Final validation

In [9]:
# Confirm zero missing values remain
assert df.isnull().sum().sum() == 0, "Missing values remain — check above steps"
print("✓ No missing values")
print(f"✓ Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print()
df.info()

✓ No missing values
✓ Final shape: 18,828 rows × 9 columns

<class 'pandas.core.frame.DataFrame'>
Index: 18828 entries, 0 to 19118
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Year              18828 non-null  int64  
 1   Month             18828 non-null  object 
 2   State             18828 non-null  object 
 3   Location          18828 non-null  object 
 4   Food              18828 non-null  object 
 5   Species           18828 non-null  object 
 6   Illnesses         18828 non-null  int64  
 7   Hospitalizations  18828 non-null  float64
 8   Fatalities        18828 non-null  float64
dtypes: float64(2), int64(2), object(5)
memory usage: 1.4+ MB


In [10]:
# Sanity check: key column distributions
print("Year range:  ", df["Year"].min(), "–", df["Year"].max())
print("States:      ", df["State"].nunique(), "unique values")
print("Food:        ", df["Food"].nunique(), "unique values (incl. 'Unknown')")
print("Species:     ", df["Species"].nunique(), "unique values (incl. 'Unknown')")
df.describe()

Year range:   1998 – 2015
States:       55 unique values
Food:         3128 unique values (incl. 'Unknown')
Species:      201 unique values (incl. 'Unknown')


,Year,Illnesses,Hospitalizations,Fatalities
count,18828.000000,18828.000000,18828.000000,18828.000000
mean,2005.586733,19.762375,0.778840,0.017846
std,5.158699,49.788507,4.832847,0.351340
min,1998.000000,2.000000,0.000000,0.000000
25%,2001.000000,3.000000,0.000000,0.000000
50%,2005.000000,8.000000,0.000000,0.000000
75%,2010.000000,19.000000,0.000000,0.000000
max,2015.000000,1939.000000,308.000000,33.000000


## 7. Export

In [11]:
df.to_csv("cleaned_data.csv", index=False)
print("Saved: cleaned_data.csv")
print(f"  → {df.shape[0]:,} records × {df.shape[1]} columns")
print()
print("Next step: 02_entity_normalisation.ipynb")
print("  Food column:   3,128 unique strings → fuzzy grouping + manual mapping")
print("  Species column: 201 unique strings → fuzzy grouping + manual mapping")

Saved: cleaned_data.csv
  → 18,828 records × 9 columns

Next step: 02_entity_normalisation.ipynb
  Food column:   3,128 unique strings → fuzzy grouping + manual mapping
  Species column: 201 unique strings → fuzzy grouping + manual mapping
